In [7]:
import pandas as pd
df = pd.read_csv("../csv/capcut_reviews_raw.csv")
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,lang
0,95c4c332-22f7-4a3e-9049-83ff9ee11a8b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jangan jadi pro semua dong balikin versi gak p...,1,0,17.2.0,2026-03-25 12:35:32,NaN,NaN,17.2.0,id
1,5ac7bfec-8739-497b-bf98-0d56b6c4dd73,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,sekolah SD 4 darul aman,5,0,NaN,2026-03-25 12:35:25,NaN,NaN,NaN,id
2,6f34c887-fe3e-4f20-85b8-d7a77d898eb0,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,saya si tdk rekomendasi aplikasi capcut ini ha...,1,0,17.2.0,2026-03-25 12:29:58,NaN,NaN,17.2.0,id
3,29d9d210-06ae-4c46-a3f8-828cf011bb31,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,saya coba dulu,4,0,NaN,2026-03-25 12:28:15,NaN,NaN,NaN,id
4,e682a46c-5d79-4b82-9ad2-40307d56d2d6,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,ah- ahh ah mphh a-ah,1,0,17.2.0,2026-03-25 12:26:53,NaN,NaN,17.2.0,id


In [8]:
df["at"] = pd.to_datetime(df["at"]) #ubah jadi datetime
df = df[["reviewId", "content", "score", "at", "lang"]]

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   reviewId  10000 non-null  object        
 1   content   9999 non-null   object        
 2   score     10000 non-null  int64         
 3   at        10000 non-null  datetime64[ns]
 4   lang      10000 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 390.8+ KB


,reviewId,content,score,at,lang
0,95c4c332-22f7-4a3e-9049-83ff9ee11a8b,jangan jadi pro semua dong balikin versi gak p...,1,2026-03-25 12:35:32,id
1,5ac7bfec-8739-497b-bf98-0d56b6c4dd73,sekolah SD 4 darul aman,5,2026-03-25 12:35:25,id
2,6f34c887-fe3e-4f20-85b8-d7a77d898eb0,saya si tdk rekomendasi aplikasi capcut ini ha...,1,2026-03-25 12:29:58,id
3,29d9d210-06ae-4c46-a3f8-828cf011bb31,saya coba dulu,4,2026-03-25 12:28:15,id
4,e682a46c-5d79-4b82-9ad2-40307d56d2d6,ah- ahh ah mphh a-ah,1,2026-03-25 12:26:53,id


In [9]:
print(f"Jumlah review bahasa Indonesia: {len(df)}")
df["score"].value_counts().sort_index() # buat lihat persebaran untuk bintangnya

Jumlah review bahasa Indonesia: 10000


score
1    3158
2     718
3     796
4     957
5    4371
Name: count, dtype: int64

In [11]:
def label_sentiment(score):
    if score <= 2:
        return "negatif"
    elif score == 3:
        return "netral"
    else:
        return "positif"
df["sentiment"] = df["score"].apply(label_sentiment)
df["sentiment"].value_counts()

sentiment
positif    5328
negatif    3876
netral      796
Name: count, dtype: int64

Labelling "positif" untuk review score di atas 3 dan "negatif" untuk score di bawah 3

In [12]:
import re
import string
import unicodedata

def normalize_text(text):
    return unicodedata.normalize('NFKD', str(text))

def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F900-\U0001F9FF"
        u"\U0001FA00-\U0001FAFF"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df["clean_text"] = df["content"].apply(normalize_text)
df["clean_text"] = df["clean_text"].apply(remove_emoji)
df["clean_text"] = df["clean_text"].str.lower()
df["clean_text"] = df["clean_text"].apply(remove_punctuation)

df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"http\S+|www\S+", "", x)
)

# FIXED LINE
df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"[^\w\s]", " ", x)
)

df["clean_text"] = df["clean_text"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

df["clean_text"] = df["clean_text"].fillna("")

#compare
df[["content", "clean_text"]].head(10)

,content,clean_text
0,jangan jadi pro semua dong balikin versi gak p...,jangan jadi pro semua dong balikin versi gak p...
1,sekolah SD 4 darul aman,sekolah sd 4 darul aman
2,saya si tdk rekomendasi aplikasi capcut ini ha...,saya si tdk rekomendasi aplikasi capcut ini ha...
3,saya coba dulu,saya coba dulu
4,ah- ahh ah mphh a-ah,ah ahh ah mphh aah
5,kikir!,kikir
6,"aplikasi nya sangat baguss,saya sangat suka!!",aplikasi nya sangat bagusssaya sangat suka
7,Ter update,ter update
8,gw pake ini kalau bikin slowmo patah patah yan...,gw pake ini kalau bikin slowmo patah patah yan...
9,good,good


Cleaning content dari emoji, fancy characters, over-punctuations, whitespace, dan URL. Lalu merubah semua characters ke lowercase untuk standarisasi.

In [13]:
slang_dict = {
    "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak",
    "gk": "tidak", "tdk": "tidak", "g": "tidak",
    "sy": "saya", "aku": "saya", "gue": "saya", "gw": "saya",
    "lu": "kamu", "lo": "kamu", "elu": "kamu",
    "km": "kamu", "kamu": "kamu",
    "udh": "sudah", "udah": "sudah", "dah": "sudah",
    "blm": "belum", "blum": "belum",
    "bgt": "banget", "bngt": "banget",
    "yg": "yang", "yng": "yang",
    "dgn": "dengan", "dg": "dengan",
    "utk": "untuk", "tuk": "untuk",
    "jg": "juga", "jga": "juga",
    "sdh": "sudah", "sdng": "sedang",
    "krn": "karena", "karna": "karena",
    "bisa": "bisa", "bs": "bisa",
    "dr": "dari", "dri": "dari",
    "di": "di", "dlm": "dalam",
    "emg": "memang", "emang": "memang",
    "gimana": "bagaimana", "gmna": "bagaimana",
    "aja": "saja", "aj": "saja",
    "dpt": "dapat", "dpat": "dapat",
    "mau": "mau", "mo": "mau",
    "skrg": "sekarang", "skrang": "sekarang",
    "trs": "terus", "trus": "terus",
    "pengen": "ingin", "pgn": "ingin",
    "klo": "kalau", "kl": "kalau", "kalo": "kalau",
    "tp": "tapi", "tpi": "tapi",
    "apk": "aplikasi", "app": "aplikasi",
    "lg": "lagi", "lgi": "lagi",
    "nih": "ini", "ni": "ini",
    "itu": "itu", "tu": "itu",
    "hbs": "habis", "abis": "habis",
    "jgn": "jangan",
    "msh": "masih", "msih": "masih",
    "bnyk": "banyak", "bnyak": "banyak",
    "pke": "pakai", "pkai": "pakai", "make": "pakai",
    "sampe": "sampai", "smpe": "sampai",
    "knp": "kenapa", "knapa": "kenapa",
    "napa": "kenapa",
}

def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return " ".join(normalized)

df["clean_text"] = df["clean_text"].apply(normalize_slang)
df[["content", "clean_text"]].head(10)

,content,clean_text
0,jangan jadi pro semua dong balikin versi gak p...,jangan jadi pro semua dong balikin versi tidak...
1,sekolah SD 4 darul aman,sekolah sd 4 darul aman
2,saya si tdk rekomendasi aplikasi capcut ini ha...,saya si tidak rekomendasi aplikasi capcut ini ...
3,saya coba dulu,saya coba dulu
4,ah- ahh ah mphh a-ah,ah ahh ah mphh aah
5,kikir!,kikir
6,"aplikasi nya sangat baguss,saya sangat suka!!",aplikasi nya sangat bagusssaya sangat suka
7,Ter update,ter update
8,gw pake ini kalau bikin slowmo patah patah yan...,saya pake ini kalau bikin slowmo patah patah y...
9,good,good


In [17]:
import nltk
from nltk.tokenize import word_tokenize

def tokenize_text(text):
    tokens = word_tokenize(text)
    return tokens

#Terapkan tokenisasi pada kolom content_expanded
df['tokenized_content'] = df['clean_text'].astype(str).apply(tokenize_text)

df[['content', 'clean_text', 'tokenized_content']].head()

,content,clean_text,tokenized_content
0,jangan jadi pro semua dong balikin versi gak p...,jangan jadi pro semua dong balikin versi tidak...,"[jangan, jadi, pro, semua, dong, balikin, vers..."
1,sekolah SD 4 darul aman,sekolah sd 4 darul aman,"[sekolah, sd, 4, darul, aman]"
2,saya si tdk rekomendasi aplikasi capcut ini ha...,saya si tidak rekomendasi aplikasi capcut ini ...,"[saya, si, tidak, rekomendasi, aplikasi, capcu..."
3,saya coba dulu,saya coba dulu,"[saya, coba, dulu]"
4,ah- ahh ah mphh a-ah,ah ahh ah mphh aah,"[ah, ahh, ah, mphh, aah]"


Cleaning slangs

In [20]:
# 1. Inisialisasi Stopwords Default
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stop_factory = StopWordRemoverFactory()
more_stopword = ['dengan', 'ia', 'bahwa', 'oleh']
stop_words = set(stop_factory.get_stop_words() + more_stopword)

# 2. Pertahankan Kata Negasi
important_words = {"tidak", "nggak", "gak", "gk", "tapi", "tp", "tpi"}

# 3. Daftar Stopwords Kustom Lengkap
custom_stopwords = [
    # umum
    "dan", "yang", "di", "ke", "dari", "untuk", "ini", "itu", "dengan", "pada",
    "adalah", "juga", "karena", "sebagai", "oleh", "atau", "dalam", "sudah",
    "belum", "saya", "aku", "kami", "kita", "kamu", "dia", "mereka", "anda", "saya", "sy", "aku", "ak"

    # informal / singkatan
    "yg", "dgn", "dr", "dlm", "aja", "kok", "sih", "deh", "dong", "nih",
    "nya", "lah", "pun", "kan", "tuh", "ga", "gak", "nggak", "tidak", "gk", "g", "tdk", "ndak"

    # kata kerja umum (sering tidak bermakna kuat)
    "ada", "jadi", "buat", "bikin", "bisa", "dapat", "harus", "ingin",
    "mau", "perlu", "coba", "pakai", "gunakan", "dipakai", "digunakan", "pake", "makai", "pke"

    # konteks aplikasi / review
    "aplikasi", "app", "apk", "capcut", "fitur", "fiturnya", "update",
    "versi", "developer", "pengembang", "download", "install",
    "uninstall", "login", "logout",

    # kata sifat umum (kurang informatif)
    "bagus", "jelek", "baik", "buruk", "keren", "mantap",
    "parah", "lumayan", "biasa", "standar",

    # kata umum review
    "banget", "sekali", "cukup", "terlalu", "lebih", "kurang",
    "masih", "lagi", "terus", "malah", "hanya", "cuma",

    # kata noise tambahan
    "ya", "iya", "ok", "oke", "okey", "sip", "wow",
    "hehe", "wkwk", "wk", "lol", "haha"
]

# 4. Konstruksi Filter Final
# (Default dikurangi kata negasi) + Kustom
filtered_stop_words = (stop_words - important_words)
filtered_stop_words.update(custom_stopwords)

# 5. Fungsi Pembersihan (Input: tokens yang sudah lowercase)
def remove_stopwords(tokens):
    return [
        token.strip()
        for token in tokens
        if token not in filtered_stop_words and len(token) > 2
    ]

# 6. Terpakan pada kolom 'tokens_lower'
df["tokens_cleaned"] = df["tokenized_content"].apply(remove_stopwords)

# Verifikasi Hasil
display(df[["tokenized_content", "tokens_cleaned"]].head())
print(f"Total Stopwords yang diterapkan: {len(filtered_stop_words)}")


,tokenized_content,tokens_cleaned
0,"[jangan, jadi, pro, semua, dong, balikin, vers...","[jangan, pro, semua, balikin, pronya]"
1,"[sekolah, sd, 4, darul, aman]","[sekolah, darul, aman]"
2,"[saya, si, tidak, rekomendasi, aplikasi, capcu...","[rekomendasi, aplikasi, berlangganan, dulu, co..."
3,"[saya, coba, dulu]",[dulu]
4,"[ah, ahh, ah, mphh, aah]","[ahh, mphh, aah]"


Total Stopwords yang diterapkan: 201


In [21]:
df.to_csv("../csv/capcut_reviews_cleaned.csv", index=False)